In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from sklearn.metrics import r2_score

import os
import sys

sys.path.append('/Users/kjesta/Desktop/Master prosjekt/Master_code/Lab/')

from funcs import *
from clean_and_combine import *

import matplotlib.style as mplstyle
mplstyle.use(["ggplot", "fast"])

import warnings
warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Scaling waves

Based of waves ran on 40 cm water depth, I will choose 3 candiates which show damping and scale them to smaller water depths by keeping the wave steepness and relative depth constant. Doing so, we can essentially run "the same wave" and adjust the relative size of the porous bed (the section of the depth consisting of plates).

## Methods used for the results:
For a given experiment of 40 cm water depth for a given frequency and amplitude I, 

1) Opened all three runs for experiments with both plates and empty tank.
2) Combined the three respective runs for experiment with plates and empty tank.
3) Subtracted the mean of the first 200 rows in the data to take out probe noise.
4) Applied a low-pass filter to smooth out outliers.
5) Took the mean of the three runs. So I was left with one dataset with plates and one without.
6) Selected 4 probes from the virtual array of 20 probes to look at. The placements of the probes chosen was then $11.075, 12.595, 13.81, 15.33$. The plates were placed between $12.4$ m and $15.45$ m.
7) Plotted the signal from the first probe, as this is not disturbed by the plates. From this plot, I visually decided at which point in time the wave train sabilized.
8) Used the timestamp of stable waves and cut off the first part of the signal for all probes. 
9) Tried to find a method for cutting off parts of the signal which contains reflected waves. With no luck. Perhaps because there were no waves that were stabilized + not reflected (due to the waves being long and the probes being placed in the middle of the tank).
10) Subtracted the mean of each probe signal from the data to again avoid gauge noise. Then found the standard deviation of each probe, and from there found the amplitude.
11) Saw how the amplitude changed from probe to probe.
12) Estimated wave number and frequency by Fourier transform. 
13) Used this to calculate the dimensionless variables, $kH$ and $ak$.

* Amplitude difference from before plates to the end of the plates: $ A_{diff} = A_{x=11.075 m} - A_{x=15.33 m}$
* Percentage damping due to plates = $\frac{A_{diff}^{plates + walls} - A_{diff}^{walls}}{A_{diff}^{plates + walls}}$

## Method for scaling

Take one original wave at depth $H_1$, with wave steepness $k_1*a_1$ and relative depth $k_1*H_1$. We want to scale this wave by keeping these two constant.
So for a new wave with depth $H_2$, we should have:

$k_1*a_1 = k_2*a_2$ and $k_1*H_1 = k_2*H_2$.

So,
$$k_2 = \frac{k_1 H_1}{H_2}$$
$$a_2 = \frac{k_1 a_1}{k_2} = \frac{a_1 H_2}{H_1}$$

And thus for the dispersion relation,
$$\omega_1^2 = gk_1 tanh(k_1 H_1)$$
$$\omega_2^2 = gk_2 tanh(k_2 H_2)$$

but with $k_1*H_1 = k_2*H_2$ we have that $tanh(k_1 H_1) = tanh(k_2 H_2)$.

So,
$$\omega_2^2 = g k_2 \frac{\omega_1^2}{g k_1} = \frac{k_2}{k_1}\omega_1^2$$
$$\omega_2 = \omega_1 \sqrt{\frac{k_2}{k_1}} = \omega_1 \sqrt{\frac{H_1}{H_2}}$$

### Reminder of wave properties
* The wavelength is $\lambda = \frac{2 \pi}{k}$
* Wave period is $T = \frac{2 \pi}{\omega} = \frac{1}{f}$
* Frequency is $f = \frac{1}{T}$
* Angular frequency $\omega = 2\pi f$
* Phase speed, $c = \frac{\omega}{k} = f \lambda$

### Results (derivations in notebooks: `\finding_waves\40cm_fXX_AXX.ipynb`)

In [19]:
# 40 cm, frequency 1 Hz, Amplitude 0.3 V

H = 0.4
a = 0.018
f = 1
w = 2 * np.pi * f
k = 2*np.pi/1.46
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 40 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 20 cm water depth---
H_new = 0.2
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 40 cm to 20 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 1.000 Hz
Original wave number: 4.304 rad/m
Original wavelength: 1.460 m
Original amplitude: 0.018 m
Original kH: 1.721
Original ak: 0.077
-----
Scaling results from 40 cm to 30 cm water depth:

New frequency: 1.155 Hz
New wave number: 5.738 rad/m
New wavelength: 1.095 m
New amplitude: 0.013 m
New kH: 1.721
New ak: 0.077
-----
Scaling results from 40 cm to 20 cm water depth:

New frequency: 1.414 Hz
New wave number: 8.607 rad/m
New wavelength: 0.730 m
New amplitude: 0.009 m
New kH: 1.721
New ak: 0.077


In [20]:
# 40 cm, frequency 0.9 Hz, Amplitude 0.3 V

H = 0.4
a = 0.016
f = 0.9
w = 2 * np.pi * f
k = 2*np.pi/1.71
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 40 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 20 cm water depth---
H_new = 0.2
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 40 cm to 20 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 0.900 Hz
Original wave number: 3.674 rad/m
Original wavelength: 1.710 m
Original amplitude: 0.016 m
Original kH: 1.470
Original ak: 0.059
-----
Scaling results from 40 cm to 30 cm water depth:

New frequency: 1.039 Hz
New wave number: 4.899 rad/m
New wavelength: 1.282 m
New amplitude: 0.012 m
New kH: 1.470
New ak: 0.059
-----
Scaling results from 40 cm to 20 cm water depth:

New frequency: 1.273 Hz
New wave number: 7.349 rad/m
New wavelength: 0.855 m
New amplitude: 0.008 m
New kH: 1.470
New ak: 0.059


In [21]:
# 40 cm, frequency 0.8 Hz, Amplitude 0.3 V

H = 0.4
a = 0.014
f = 0.8
w = 2 * np.pi * f
k = 2*np.pi/2.03
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 40 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 20 cm water depth---
H_new = 0.2
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 40 cm to 20 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 0.800 Hz
Original wave number: 3.095 rad/m
Original wavelength: 2.030 m
Original amplitude: 0.014 m
Original kH: 1.238
Original ak: 0.043
-----
Scaling results from 40 cm to 30 cm water depth:

New frequency: 0.924 Hz
New wave number: 4.127 rad/m
New wavelength: 1.522 m
New amplitude: 0.010 m
New kH: 1.238
New ak: 0.043
-----
Scaling results from 40 cm to 20 cm water depth:

New frequency: 1.131 Hz
New wave number: 6.190 rad/m
New wavelength: 1.015 m
New amplitude: 0.007 m
New kH: 1.238
New ak: 0.043


In [22]:
# 40 cm, frequency 0.7 Hz, Amplitude 0.3 V

H = 0.4
a = 0.012
f = 0.7
w = 2 * np.pi * f
k = 2*np.pi/2.43
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 40 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 20 cm water depth---
H_new = 0.2
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 40 cm to 20 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 0.700 Hz
Original wave number: 2.586 rad/m
Original wavelength: 2.430 m
Original amplitude: 0.012 m
Original kH: 1.034
Original ak: 0.031
-----
Scaling results from 40 cm to 30 cm water depth:

New frequency: 0.808 Hz
New wave number: 3.448 rad/m
New wavelength: 1.823 m
New amplitude: 0.009 m
New kH: 1.034
New ak: 0.031
-----
Scaling results from 40 cm to 20 cm water depth:

New frequency: 0.990 Hz
New wave number: 5.171 rad/m
New wavelength: 1.215 m
New amplitude: 0.006 m
New kH: 1.034
New ak: 0.031


In [23]:
# 40 cm, frequency 0.6 Hz, Amplitude 0.3 V

H = 0.4
a = 0.01
f = 0.6
w = 2 * np.pi * f
k = 2*np.pi/2.95
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 40 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 20 cm water depth---
H_new = 0.2
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 40 cm to 20 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 0.600 Hz
Original wave number: 2.130 rad/m
Original wavelength: 2.950 m
Original amplitude: 0.010 m
Original kH: 0.852
Original ak: 0.021
-----
Scaling results from 40 cm to 30 cm water depth:

New frequency: 0.693 Hz
New wave number: 2.840 rad/m
New wavelength: 2.212 m
New amplitude: 0.007 m
New kH: 0.852
New ak: 0.021
-----
Scaling results from 40 cm to 20 cm water depth:

New frequency: 0.849 Hz
New wave number: 4.260 rad/m
New wavelength: 1.475 m
New amplitude: 0.005 m
New kH: 0.852
New ak: 0.021


In [ ]:
# 40 cm, frequency 0.5 Hz, Amplitude 0.3 V

H = 0.4
a = 0.0078
f = 0.5
w = 2 * np.pi * f
k = 2*np.pi/3.73
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 40 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 20 cm water depth---
H_new = 0.2
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 40 cm to 20 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 0.500 Hz
Original wave number: 1.685 rad/m
Original wavelength: 3.730 m
Original amplitude: 0.008 m
Original kH: 0.674
Original ak: 0.013
-----
Scaling results from 40 cm to 30 cm water depth:

New frequency: 0.577 Hz
New wave number: 2.246 rad/m
New wavelength: 2.797 m
New amplitude: 0.006 m
New kH: 0.674
New ak: 0.013
-----
Scaling results from 40 cm to 20 cm water depth:

New frequency: 0.707 Hz
New wave number: 3.369 rad/m
New wavelength: 1.865 m
New amplitude: 0.004 m
New kH: 0.674
New ak: 0.013


### Results (derivations in notebooks: `\finding_waves\20cm_fXX_AXX.ipynb`)